# D0a · held-out fetch (pilot D) · CPU · Internet ON

**What it does.** It downloads everything external that D0b needs, into `/kaggle/working/d0a/`:
1. **Task pool.** It clones the repos in `pilot_d/manifest.json` from public GitHub, mines them, and builds every
   task's snapshot (`pool/snapshots/<id>.tgz`, `pool/tasks.jsonl`, `pool/meta.jsonl`).
2. **Issue text** from the GitHub API, including issues linked through a PR body's "Fixes #N"
   (`pool/tasks_text.jsonl`, `pool/meta_text.jsonl`).
3. **Wheels for D0b's offline install** (`wheels/`):
   - the codegraph-loc wheel at the pinned commit;
   - every distribution that `pip install --dry-run` reports as missing in this image for sentence-transformers and the
     declared test dependencies (`need.txt`, exact pins).
4. **Test-only wheels** for Python 3.12 and 3.13, shipped inside the export (`export_wheels/`).
5. **The `BAAI/bge-small-en-v1.5` model files** (safetensors; no remote code) in `models/`.
6. **`D0A_MANIFEST.json`**, with sha256 per file.

Before finishing, it checks itself: it installs from its own `wheels/` with `--no-index` and embeds one string with the
model fully offline.

| | |
|---|---|
| Accelerator | **CPU** (no GPU quota) |
| Internet | **ON** (GitHub clone + API, PyPI, Hugging Face) |
| Environment | the official starter's **pinned environment (Python 3.12)**, the *same* one D0b uses, so the list of missing packages matches. Open the starter, Copy & Edit, **remove the competition input**, File → Import Notebook with this file, keep "Pin to original environment", Settings → Internet ON. The first code cell stops on any other Python. |
| Attach | nothing |
| Optional | Kaggle Secret `GITHUB_TOKEN`. Without it the GitHub API allows 60 requests/h, so ~150 requests wait for 2 hourly resets. |
| Expected run time | ≈ 1–3.5 h (build ≈ 40 min; issue text ≈ 5 min with a token, ≤ 3 h without; wheels and model ≈ 5 min) |
| Output | `/kaggle/working/d0a/` (≈ 7–9 GB, < 250 files). D0b attaches this notebook's output. Bring back `d0a/D0a_summary.json` → `results/` on pilot/d-heldout. |


In [ ]:
import sys
assert sys.version_info[:2] == (3, 12), f"Python {sys.version.split()[0]}: use the starter's pinned environment (3.12)"
print(sys.version)

In [ ]:
import glob, hashlib, importlib.util, json, os, shutil, subprocess, sys, time
from pathlib import Path
OUT = Path('/kaggle/working/d0a'); TMP = Path('/tmp/d0a'); PD = TMP / 'pd'
for d in (OUT / 'pool', OUT / 'wheels', OUT / 'export_wheels', OUT / 'models', PD): d.mkdir(parents=True, exist_ok=True)
PIP = [sys.executable, '-m', 'pip']
DRY = os.environ.get('D0A_DRYRUN') == '1'   # local papermill dry run only: 2 tasks per repo, no GitHub API
t_start = time.time()

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/common', exist_ok=True)
open('/tmp/d0a/pd/common/__init__.py', 'w').write(''); print('wrote common/__init__.py')

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/common', exist_ok=True)
open('/tmp/d0a/pd/common/mining.py', 'w').write('"""Mine candidate bug-fix tasks from a local git clone (no GitHub API needed).\n\nEvery filter step is counted so yield can be reported. Test/config classification mirrors\nswegemma.harness.verification._is_protected_test_or_config_path, because that is what the\nscorer strips from agent patches and resets before applying test_patch.\n\nUsage:\n  python -I common/mining.py REPO_DIR --repo owner/name --pkg PKGDIR --since 2026-05-01 \\\n      --out CANDIDATES.jsonl --funnel FUNNEL.json [--max-code-files 3] [--max-code-lines 200]\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport subprocess\nfrom pathlib import PurePosixPath\n\nPROTECTED_CONFIG = {"conftest.py", "pytest.ini", "pyproject.toml", "tox.ini", "setup.cfg", ".pytest.ini",\n                    "sitecustomize.py", "usercustomize.py", "_swegemma_stubs.py"}\nFIX_RE = re.compile(r"^(fix|bug|bugfix|hotfix)\\b|^BUG\\b|^fix\\(", re.I)\nISSUE_REF_RE = re.compile(r"\\b(?:fix(?:es|ed)?|close[sd]?|resolve[sd]?)\\s*:?\\s+#(\\d+)", re.I)\nTRAILING_PR_RE = re.compile(r"\\(#(\\d+)\\)\\s*$")\nDOUBLE_REF_RE = re.compile(r"\\(#(\\d+)\\)\\s*\\(#(\\d+)\\)\\s*$")  # sqlglot style: (#issue) (#PR)\nTEST_ISSUE_RE = re.compile(r"(?:\\bgh-|github\\.com/[\\w.-]+/[\\w.-]+/issues/|\\bissue\\s+#?)(\\d{2,6})", re.I)\nAI_RE = re.compile(r"\\[CODEX\\]|Co-authored-by:.*(Claude|Copilot|Codex|Devin|cursor)|Generated with \\[?Claude|"\n                   r"copilot-swe-agent|\\bAI-assisted\\b", re.I)\n\n\ndef is_test_path(p: str) -> bool:\n    name = PurePosixPath(p).name\n    parts = PurePosixPath(p).parts\n    if not p.endswith(".py"):\n        return any(d in ("tests", "test", "testing") for d in parts[:-1])  # fixtures/data under tests/\n    return (name.startswith("test_") or name.endswith("_test.py")\n            or any(d in ("tests", "test", "testing") for d in parts[:-1]))\n\n\ndef is_protected_config(p: str) -> bool:\n    name = PurePosixPath(p).name\n    return name in PROTECTED_CONFIG or name.endswith(".pth")\n\n\ndef git(repo: str, *args: str) -> str:\n    return subprocess.run(["git", "-C", repo, *args], capture_output=True, text=True, check=True).stdout\n\n\ndef to_dev_style(diff: str) -> str:\n    """Rewrite `git diff` output into the dev set\'s style: only \'--- a/X\' / \'+++ b/X\' headers,\n    \'--- a/X\' even for new files, \'+++ b/X\' even for deleted files, no trailing newline."""\n    out, cur_old, cur_new = [], None, None\n    for line in diff.splitlines():\n        if line.startswith("diff --git "):\n            m = re.match(r"diff --git a/(.+) b/(.+)$", line)\n            cur_old, cur_new = m.group(1), m.group(2)\n            continue\n        if line.startswith(("index ", "new file mode", "deleted file mode", "similarity index",\n                            "rename from", "rename to", "old mode", "new mode")):\n            continue\n        if line.startswith("--- "):\n            out.append(f"--- a/{cur_old}")\n            continue\n        if line.startswith("+++ "):\n            out.append(f"+++ b/{cur_new}")\n            continue\n        out.append(line)\n    return "\\n".join(out)\n\n\n_PY_ATTR = None\n\n\ndef _python_diff_attributes() -> str:\n    """A gitattributes file enabling git\'s built-in Python diff driver, so hunk headers name the enclosing\n    `def`/`class` (e.g. \'@@ ... @@ def test_sqlite(self):\') as in the dev tasks (527 of 720 dev test-patch hunk\n    headers are `def ...`). This matters: when a test patch adds no `def test_*`, the scorer takes the required\n    tests from these headers (verification._extract_test_functions_from_patch)."""\n    global _PY_ATTR\n    if _PY_ATTR is None:\n        import tempfile\n        fd, _PY_ATTR = tempfile.mkstemp(prefix="pyattr_")\n        with open(fd, "w") as fh:\n            fh.write("*.py diff=python\\n")\n    return _PY_ATTR\n\n\ndef file_diff(repo: str, parent: str, commit: str, paths: list[str]) -> str:\n    if not paths:\n        return ""\n    return subprocess.run(["git", "-c", f"core.attributesFile={_python_diff_attributes()}", "-C", repo, "diff",\n                           "--no-color", "--no-renames", "-U3", parent, commit, "--", *paths],\n                          capture_output=True, text=True, check=True).stdout\n\n\ndef mine(args) -> None:\n    repo = args.repo_dir\n    shas = git(repo, "log", "--first-parent", f"--since={args.since}", "--format=%H", args.branch).split()\n    funnel = {"commits_since": len(shas)}\n    steps = ["single_parent", "fix_message", "has_test_and_code_change", "has_py_test_file", "pkg_python_only",\n             "no_protected_config_in_code", "size_ok"]\n    for s in steps:\n        funnel[s] = 0\n    funnel["with_issue_ref_in_message"] = 0\n    funnel["ai_authored_marker"] = 0\n    out = open(args.out, "w")\n    short = args.repo.split("/")[1].lower()\n    for sha in shas:\n        parents = git(repo, "log", "-1", "--format=%P", sha).split()\n        if len(parents) != 1:\n            continue\n        funnel["single_parent"] += 1\n        subject = git(repo, "log", "-1", "--format=%s", sha).strip()\n        body = git(repo, "log", "-1", "--format=%b", sha)\n        if not FIX_RE.search(subject):\n            continue\n        funnel["fix_message"] += 1\n        parent = parents[0]\n        names = [l.split("\\t") for l in git(repo, "diff", "--no-renames", "--name-status", parent, sha).splitlines()]\n        files = [n[-1] for n in names]\n        test_files = [f for f in files if is_test_path(f)]\n        code_files = [f for f in files if not is_test_path(f)]\n        code_py = [f for f in code_files if f.endswith(".py")]\n        if not test_files or not code_py:\n            continue\n        funnel["has_test_and_code_change"] += 1\n        # The harness runs pytest on the .py files of test_patch; fixture-only test patches\n        # make it pass a non-.py path to pytest ("no match"), so require a .py test file.\n        if not any(f.endswith(".py") and PurePosixPath(f).name != "conftest.py" for f in test_files):\n            continue\n        funnel["has_py_test_file"] += 1\n        # Code change must be Python inside the package; allow docs/changelog alongside (dropped from gold).\n        pkg_files = [f for f in code_files if f.startswith(args.pkg + "/")]\n        if any(not f.endswith(".py") for f in pkg_files) or not all(f in pkg_files for f in code_py):\n            continue\n        funnel["pkg_python_only"] += 1\n        if any(is_protected_config(f) for f in code_files):\n            continue\n        funnel["no_protected_config_in_code"] += 1\n        code_diff = file_diff(repo, parent, sha, pkg_files)\n        added = sum(1 for l in code_diff.splitlines() if l.startswith(("+", "-")) and not l.startswith(("+++", "---")))\n        if len(pkg_files) > args.max_code_files or added > args.max_code_lines:\n            continue\n        funnel["size_ok"] += 1\n        m2 = DOUBLE_REF_RE.search(subject)\n        issue_refs = sorted({int(x) for x in ISSUE_REF_RE.findall(subject + "\\n" + body)}\n                            | ({int(m2.group(1))} if m2 else set()))\n        pr = int(m2.group(2)) if m2 else (int(TRAILING_PR_RE.search(subject).group(1)) if TRAILING_PR_RE.search(subject) else None)\n        issue_refs = [i for i in issue_refs if i != pr]\n        test_diff = file_diff(repo, parent, sha, test_files)\n        added_test_text = "\\n".join(l for l in test_diff.splitlines() if l.startswith("+"))\n        issue_refs_in_tests = sorted({int(x) for x in TEST_ISSUE_RE.findall(added_test_text)} - {pr})\n        funnel["with_issue_ref_in_tests"] = funnel.get("with_issue_ref_in_tests", 0) + bool(issue_refs_in_tests)\n        funnel["with_any_issue_ref"] = funnel.get("with_any_issue_ref", 0) + bool(issue_refs or issue_refs_in_tests)\n        funnel["with_issue_ref_in_message"] += bool(issue_refs)\n        ai = bool(AI_RE.search(subject + "\\n" + body))\n        funnel["ai_authored_marker"] += ai\n        rec = {\n            "instance_id": f"{short}_{pr or sha[:10]}",\n            "repo": args.repo,\n            "base_commit": parent,\n            "fix_commit": sha,\n            "pr_number": pr,\n            "issue_refs": issue_refs,\n            "issue_refs_in_tests": issue_refs_in_tests,\n            "subject": subject,\n            "committed_at": git(repo, "log", "-1", "--format=%cI", sha).strip(),\n            "authored_at": git(repo, "log", "-1", "--format=%aI", sha).strip(),\n            "code_files": pkg_files,\n            "test_files": test_files,\n            "code_lines_changed": added,\n            "ai_authored_marker": ai,\n            "patch": to_dev_style(code_diff),\n            "test_patch": to_dev_style(test_diff),\n        }\n        out.write(json.dumps(rec) + "\\n")\n    json.dump(funnel, open(args.funnel, "w"), indent=1)\n    print(json.dumps(funnel))\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser()\n    p.add_argument("repo_dir")\n    p.add_argument("--repo", required=True)\n    p.add_argument("--pkg", required=True)\n    p.add_argument("--since", default="2026-05-01")\n    p.add_argument("--branch", default="HEAD")\n    p.add_argument("--out", required=True)\n    p.add_argument("--funnel", required=True)\n    p.add_argument("--max-code-files", type=int, default=3)\n    p.add_argument("--max-code-lines", type=int, default=200)\n    mine(p.parse_args())\n'); print('wrote common/mining.py')

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/common', exist_ok=True)
open('/tmp/d0a/pd/common/snapshot.py', 'w').write('"""Build a task snapshot .tgz in the same layout as the competition\'s snapshots/<instance_id>.tgz.\n\nObserved dev layout (schema only): archive rooted at \'./\', a git repo whose history is\n`git fast-export` of the source up to base_commit piped into `git fast-import` (so commit hashes\ndiffer from upstream; tasks.jsonl keeps the upstream base_commit), branches `export_ref` and\n`main` (checked out) at the base, .git/config with user agent@eval / Agent, clean working tree.\nHistory depth is whatever the source clone has (dev snapshots range from 4 to 6k commits).\n\nUsage: python -I common/snapshot.py REPO_DIR BASE_COMMIT OUT.tgz\n"""\nfrom __future__ import annotations\n\nimport subprocess\nimport sys\nimport tempfile\nfrom pathlib import Path\n\n\ndef run(*cmd: str, cwd: str | None = None, stdin=None, stdout=None) -> None:\n    subprocess.run(cmd, cwd=cwd, check=True, stdin=stdin, stdout=stdout, stderr=subprocess.PIPE)\n\n\ndef build_snapshot(repo_dir: str, base_commit: str, out_tgz: str) -> dict:\n    out = Path(out_tgz)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory() as td:\n        dst = Path(td) / "ws"\n        dst.mkdir()\n        run("git", "init", "-q", "-b", "main", str(dst))\n        run("git", "-C", repo_dir, "branch", "-f", "export_ref", base_commit)\n        exp = subprocess.Popen(["git", "-C", repo_dir, "fast-export", "--signed-tags=strip",\n                                "--tag-of-filtered-object=drop", "--reencode=yes", "refs/heads/export_ref"],\n                               stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)\n        run("git", "-C", str(dst), "fast-import", "--quiet", stdin=exp.stdout)\n        if exp.wait() != 0:\n            raise RuntimeError("git fast-export failed")\n        run("git", "-C", str(dst), "checkout", "-q", "-B", "main", "export_ref")\n        run("git", "-C", str(dst), "config", "user.email", "agent@eval")\n        run("git", "-C", str(dst), "config", "user.name", "Agent")\n        run("git", "-C", str(dst), "gc", "-q", "--prune=now")\n        # Same top-level shape as dev archives: entries prefixed with \'./\'\n        run("tar", "-C", str(dst), "-czf", str(out), ".")\n        n_commits = int(subprocess.run(["git", "-C", str(dst), "rev-list", "--count", "HEAD"],\n                                       capture_output=True, text=True).stdout.strip() or 0)\n    return {"snapshot_bytes": out.stat().st_size, "history_commits": n_commits}\n\n\nif __name__ == "__main__":\n    print(build_snapshot(sys.argv[1], sys.argv[2], sys.argv[3]))\n'); print('wrote common/snapshot.py')

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/common', exist_ok=True)
open('/tmp/d0a/pd/common/github_text.py', 'w').write('"""Fill problem_statement / created_at from GitHub (issue first, PR body as fallback).\n\nRun where api.github.com is reachable (Kaggle notebook with Internet ON, or a laptop). Optional\nGITHUB_TOKEN (env var or Kaggle Secret) raises the limit to 5,000 req/h; unauthenticated is 60 req/h (about\n25-40 tasks per hour; the script waits for the hourly reset, bounded by GITHUB_MAX_RESET_WAIT).\n\nproblem_statement follows the SWE-bench convention: "<title>\\n<body>". created_at is the issue\'s\ncreation time (contamination dating). Provenance is recorded in meta: issue | pr_body.\n\nUsage: python common/github_text.py TASKS.jsonl META.jsonl OUT_TASKS.jsonl OUT_META.jsonl\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport re\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\n\n\ndef _token() -> str | None:\n    """GITHUB_TOKEN env var, else an optional Kaggle Secret of the same name (never required)."""\n    if os.environ.get("GITHUB_TOKEN"):\n        return os.environ["GITHUB_TOKEN"]\n    try:\n        from kaggle_secrets import UserSecretsClient  # only exists inside Kaggle notebooks\n        return UserSecretsClient().get_secret("GITHUB_TOKEN") or None\n    except Exception:\n        return None\n\n\nTOKEN = _token()\n# same-repo closing references in a PR body ("Fixes #123"); cross-repo refs (owner/repo#N) are ignored\nCLOSE_RE = re.compile(r"\\b(?:fix(?:es|ed)?|close[sd]?|resolve[sd]?)\\s*:?\\s+#(\\d+)", re.I)\nMAX_RESET_WAIT = int(os.environ.get("GITHUB_MAX_RESET_WAIT", "3700"))  # seconds; unauthenticated limit resets hourly\n\n\ndef gh(path: str) -> dict | None:\n    req = urllib.request.Request(f"https://api.github.com{path}",\n                                 headers={"Accept": "application/vnd.github+json", "User-Agent": "heldout-builder"})\n    if TOKEN:\n        req.add_header("Authorization", f"Bearer {TOKEN}")\n    for attempt in range(6):\n        try:\n            return json.loads(urllib.request.urlopen(req, timeout=30).read())\n        except urllib.error.HTTPError as e:\n            if e.code == 404:\n                return None\n            if e.code in (403, 429):\n                # primary rate limit: wait for the reset the API announces (bounded); else short backoff\n                remaining, reset = e.headers.get("x-ratelimit-remaining"), e.headers.get("x-ratelimit-reset")\n                if remaining == "0" and reset:\n                    wait = max(1, int(reset) - int(time.time()) + 2)\n                    if wait > MAX_RESET_WAIT:\n                        raise RuntimeError(f"GitHub rate limit resets in {wait}s (> {MAX_RESET_WAIT}s); "\n                                           "add a GITHUB_TOKEN Kaggle Secret or rerun later") from e\n                    print(f"rate limited; waiting {wait}s for reset", flush=True)\n                    time.sleep(wait)\n                else:\n                    time.sleep(min(60, 2 ** attempt * 5))\n                continue\n            raise\n    return None\n\n\ndef main():\n    tasks = [json.loads(l) for l in open(sys.argv[1])]\n    meta = {m["instance_id"]: m for m in map(json.loads, open(sys.argv[2]))}\n    with open(sys.argv[3], "w") as ft, open(sys.argv[4], "w") as fm:\n        for t in tasks:\n            m = meta[t["instance_id"]]\n            text, src, created = None, None, None\n            for n in list(m.get("issue_refs", [])) + list(m.get("issue_refs_in_tests", [])):\n                it = gh(f"/repos/{t[\'repo\']}/issues/{n}")\n                if it and "pull_request" not in it:  # a real issue, not a PR\n                    text, src, created = f"{it[\'title\']}\\n{it.get(\'body\') or \'\'}", f"issue#{n}", it["created_at"]\n                    break\n            if text is None and m.get("pr_number"):\n                pr = gh(f"/repos/{t[\'repo\']}/pulls/{m[\'pr_number\']}")\n                if pr:\n                    for n in dict.fromkeys(CLOSE_RE.findall(pr.get("body") or "")):  # PR body links the issue\n                        it = gh(f"/repos/{t[\'repo\']}/issues/{n}")\n                        if it and "pull_request" not in it:\n                            text, src, created = f"{it[\'title\']}\\n{it.get(\'body\') or \'\'}", f"issue#{n}", it["created_at"]\n                            m["issue_refs"] = sorted(set(m.get("issue_refs") or []) | {int(n)})\n                            break\n                    if text is None:\n                        text, src, created = f"{pr[\'title\']}\\n{pr.get(\'body\') or \'\'}", f"pr_body#{m[\'pr_number\']}", pr["created_at"]\n            if text is not None:\n                t["problem_statement"], t["created_at"] = text.replace("\\r\\n", "\\n"), created\n                m.update(needs_issue_text=False, problem_statement_source=src)\n            ft.write(json.dumps(t) + "\\n")\n            fm.write(json.dumps(m) + "\\n")\n            print(t["instance_id"], src, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote common/github_text.py')

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/pilot_d', exist_ok=True)
open('/tmp/d0a/pd/pilot_d/__init__.py', 'w').write(''); print('wrote pilot_d/__init__.py')

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/pilot_d', exist_ok=True)
open('/tmp/d0a/pd/pilot_d/build_tasks.py', 'w').write('"""Pilot D: turn mined candidates into competition-format tasks (+ sidecar metadata).\n\nWrites, under OUT_DIR (outside git):\n  tasks.jsonl            competition fields (+ optional FAIL_TO_PASS/PASS_TO_PASS later)\n  meta.jsonl             provenance: fix commit, PR, issue refs, dates, AI marker, timings\n  snapshots/<iid>.tgz    dev-layout snapshot\nproblem_statement is left EMPTY with needs_issue_text=true: issue text must be fetched from the\nGitHub API (common/github_text.py), which this cloud session cannot reach for third-party repos.\n\nUsage: python -I pilot_d/build_tasks.py CANDIDATES.jsonl REPO_DIR OUT_DIR [--n N] [--issue-linked-only]\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom common.snapshot import build_snapshot  # noqa: E402\n\n\ndef spread(rows, n):\n    """Pick n rows spread evenly over time (rows sorted by commit date)."""\n    rows = sorted(rows, key=lambda r: r["committed_at"])\n    if len(rows) <= n:\n        return rows\n    step = len(rows) / n\n    return [rows[int(i * step)] for i in range(n)]\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("candidates")\n    p.add_argument("repo_dir")\n    p.add_argument("out_dir")\n    p.add_argument("--n", type=int, default=12)\n    p.add_argument("--issue-linked-only", action="store_true")\n    p.add_argument("--ids", nargs="*", help="build exactly these instance_ids (e.g. from pilot_d/manifest.json)")\n    a = p.parse_args()\n    rows = [json.loads(l) for l in open(a.candidates)]\n    if a.ids:\n        chosen = [r for r in rows if r["instance_id"] in set(a.ids)]\n    else:\n        if a.issue_linked_only:\n            rows = [r for r in rows if r["issue_refs"]]\n        chosen = spread(rows, a.n)\n    out = Path(a.out_dir)\n    (out / "snapshots").mkdir(parents=True, exist_ok=True)\n    with open(out / "tasks.jsonl", "a") as ft, open(out / "meta.jsonl", "a") as fm:\n        for r in chosen:\n            t0 = time.perf_counter()\n            snap = build_snapshot(a.repo_dir, r["base_commit"], str(out / "snapshots" / f"{r[\'instance_id\']}.tgz"))\n            task = {\n                "instance_id": r["instance_id"], "repo": r["repo"], "base_commit": r["base_commit"],\n                "patch": r["patch"], "test_patch": r["test_patch"],\n                "problem_statement": "", "hints_text": "", "created_at": r["authored_at"],\n            }\n            meta = {k: r.get(k) for k in ("instance_id", "repo", "fix_commit", "pr_number", "issue_refs",\n                                      "issue_refs_in_tests", "subject",\n                                      "committed_at", "authored_at", "code_files", "test_files",\n                                      "code_lines_changed", "ai_authored_marker")}\n            meta.update(snap, needs_issue_text=True, build_seconds=round(time.perf_counter() - t0, 1))\n            ft.write(json.dumps(task) + "\\n")\n            fm.write(json.dumps(meta) + "\\n")\n            print(r["instance_id"], snap, meta["build_seconds"], flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote pilot_d/build_tasks.py')

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/pilot_d', exist_ok=True)
open('/tmp/d0a/pd/pilot_d/manifest.json', 'w').write('{\n "since": "2026-05-01",\n "clone_shallow_since": "2026-03-15",\n "note": "Held-out pool (pilot D). instance_id = <repo_short>_<PR number or fix-commit prefix>; public upstream commits only. Rebuilt by notebooks/D0_heldout_export__CPU.ipynb (pilot_d/build_tasks.py --ids).",\n "selection_rule": "sqlglot: fix commits with an issue reference in the message or test diff; other repos: all fix commits passing common/mining.py filters (<=3 files, <=200 changed lines)",\n "dropped_repos": {\n  "pydata/xarray": "1/12 valid locally: whole test modules skipped for missing optional backends, version-sensitive failures; environment surface too large",\n  "Python-Markdown/markdown": "extensions load by entry-point name, which needs an INSTALLED copy; the subprocess sandbox never installs the workspace (2/9 valid)"\n },\n "embeddings": {\n  "model": "bge",\n  "input": "text",\n  "map": false,\n  "why": "top-10-neighbour agreement with the released embeddings on 4 dev graphs: bge 0.174, gte 0.183 (random 0.009); bge chosen for speed (22 vs 5 texts/s on 4 CPUs, measured), needed to embed ~80 graphs on Kaggle CPU; the linear map into the released geometry lowered agreement for every model (leave-one-repo-out), so none is used"\n },\n "repos": [\n  {\n   "short": "sqlglot",\n   "repo": "tobymao/sqlglot",\n   "url": "https://github.com/tobymao/sqlglot",\n   "pkg": "sqlglot",\n   "licence": "MIT",\n   "instance_ids": [\n    "sqlglot_8856",\n    "sqlglot_8857",\n    "sqlglot_8825",\n    "sqlglot_8809",\n    "sqlglot_8686",\n    "sqlglot_8670",\n    "sqlglot_8666",\n    "sqlglot_8596",\n    "sqlglot_a00dc2e5c4",\n    "sqlglot_8523",\n    "sqlglot_f8606fe8f9",\n    "sqlglot_8538",\n    "sqlglot_b65592b458",\n    "sqlglot_8476",\n    "sqlglot_8447",\n    "sqlglot_8434",\n    "sqlglot_981e492413",\n    "sqlglot_8d9f5d08cc",\n    "sqlglot_a3884168f6",\n    "sqlglot_8369",\n    "sqlglot_3ca824895e",\n    "sqlglot_8321",\n    "sqlglot_8284",\n    "sqlglot_8285",\n    "sqlglot_8269",\n    "sqlglot_8267",\n    "sqlglot_8270",\n    "sqlglot_8260",\n    "sqlglot_40f285cb61",\n    "sqlglot_a570cf8a90",\n    "sqlglot_f64adb4c9a",\n    "sqlglot_83129c711b",\n    "sqlglot_8138",\n    "sqlglot_2f914c54aa",\n    "sqlglot_c25225d9ac",\n    "sqlglot_df5ec1ecd4",\n    "sqlglot_b4776a4099",\n    "sqlglot_d6d174e85e",\n    "sqlglot_7968",\n    "sqlglot_169b8364b6",\n    "sqlglot_a1e3338e31",\n    "sqlglot_a66104f52a",\n    "sqlglot_7809",\n    "sqlglot_4c2cd15c7a",\n    "sqlglot_6ae4b49f1d",\n    "sqlglot_a40dde9d39",\n    "sqlglot_7640",\n    "sqlglot_a1fa1546c9",\n    "sqlglot_b9f18eea67",\n    "sqlglot_17432c6117"\n   ],\n   "fix_commits": {\n    "sqlglot_8856": "6de2df31b0ae8b21155828f575a14dc8903c1b9b",\n    "sqlglot_8857": "f70f4e5be5acc50058f4d4ac1ee0f250e203916b",\n    "sqlglot_8825": "13a570c6f790a6140c9c9ae004914b081beb7ae6",\n    "sqlglot_8809": "2d4b5a5b0aea46e7f5a4fbe09c1d60fcafad76b3",\n    "sqlglot_8686": "fdd0e9a464f0f9ec425c56ae83fc87a5e2052087",\n    "sqlglot_8670": "f8c314c449080860604b9ddd716d6bc75c89155f",\n    "sqlglot_8666": "513beedd3753a22c1ce234059bd5d831f64e48f0",\n    "sqlglot_8596": "e38929edd78a20f74fc19fee819e14ac23add687",\n    "sqlglot_a00dc2e5c4": "a00dc2e5c489214bea53bacbbeba8713f21ebce1",\n    "sqlglot_8523": "3096a591d65b1b00ba9abdf482fc7b7973a6e6af",\n    "sqlglot_f8606fe8f9": "f8606fe8f97a10a3ba00f42104836add1299966f",\n    "sqlglot_8538": "e1e0d2e52f45247bfa0ab815eb8a7388ea50a0e8",\n    "sqlglot_b65592b458": "b65592b45831df5e7f13d418e50f24c4cea118f0",\n    "sqlglot_8476": "ac084c5bdb2f188ff878687000273ae2247db24b",\n    "sqlglot_8447": "b3b6c57cea516bf8f76c6262b308d86dff82ca1b",\n    "sqlglot_8434": "d2fbb215ff6bd548547a151e8838c277aad076c1",\n    "sqlglot_981e492413": "981e492413740f7740d2b0288e3da5b4028c767e",\n    "sqlglot_8d9f5d08cc": "8d9f5d08cce4b72b4df1a8f6394c2aa043cc7de8",\n    "sqlglot_a3884168f6": "a3884168f639633fa8f64334bf63e4b343bedfe2",\n    "sqlglot_8369": "65eb23b53420212c20f394d399ca18efefec90e3",\n    "sqlglot_3ca824895e": "3ca824895ef423f7895fb13d72357548c0f1f367",\n    "sqlglot_8321": "e27b7654a9d710e29da6f42994de845a99444c84",\n    "sqlglot_8284": "6b9cb87bd01ed4606d1902a0266ff79aad78b54f",\n    "sqlglot_8285": "67cf1ec91dfe2658f41c049130ab242a40bf4789",\n    "sqlglot_8269": "d0aa2b432498ec00f88a8721a904afe2e1b5c24d",\n    "sqlglot_8267": "3110e151bfe6e229ec6b85708a56a9340a56c6a6",\n    "sqlglot_8270": "f131ec9f9ea40ce55f0e254baa41b01f87bad25c",\n    "sqlglot_8260": "88419b1aa19453be7fca80c0178362b1fbf2d6e1",\n    "sqlglot_40f285cb61": "40f285cb61edff4a40950fc708e79ae4e8ed583d",\n    "sqlglot_a570cf8a90": "a570cf8a908d02e78b66cec1478ab066f1ff1f0d",\n    "sqlglot_f64adb4c9a": "f64adb4c9a3f96ff4ad692d571b80848ad1b649d",\n    "sqlglot_83129c711b": "83129c711baf600f6ae902f79cf1cd6e50bdba58",\n    "sqlglot_8138": "1dae76b5833a47c9e2bc0a72a93d94d829377a0c",\n    "sqlglot_2f914c54aa": "2f914c54aa4817a0a5e65a0d7d5e7cba9037ac30",\n    "sqlglot_c25225d9ac": "c25225d9ac7390ef87bd02a3d2ac005073a1e27d",\n    "sqlglot_df5ec1ecd4": "df5ec1ecd43b83ac63800bf40b04362c9af0173e",\n    "sqlglot_b4776a4099": "b4776a40995985c5523849269e1ab6d29e667ec6",\n    "sqlglot_d6d174e85e": "d6d174e85e37d7644d16c0bce3e4fb49384a6bc8",\n    "sqlglot_7968": "06eb0510b06307ffad115827bc7c9c53deecb216",\n    "sqlglot_169b8364b6": "169b8364b643c4285dcf073aa2f357c5b40831a6",\n    "sqlglot_a1e3338e31": "a1e3338e31354496a3ef3bed5f3c5e0c4df4e0f6",\n    "sqlglot_a66104f52a": "a66104f52aca450087524f84b76e29173971e656",\n    "sqlglot_7809": "cf7e4bf4a57d331e4a6e5f4933cb7daf0e503c3b",\n    "sqlglot_4c2cd15c7a": "4c2cd15c7a66bb240a8508517bb492aa7ed6c84a",\n    "sqlglot_6ae4b49f1d": "6ae4b49f1d2ef0c4f30575a29a0d1b3273520abf",\n    "sqlglot_a40dde9d39": "a40dde9d3992b2b1df4f4ff5dccde240d618dabd",\n    "sqlglot_7640": "63296846f6a7a683af21683acb24aad7a63bb3f5",\n    "sqlglot_a1fa1546c9": "a1fa1546c926b2858cfc5adcfbd186731de40eba",\n    "sqlglot_b9f18eea67": "b9f18eea6762f8ce03ff5a426db8195db375f455",\n    "sqlglot_17432c6117": "17432c61175194ee0f56f0703393e8c5fdc9e81f"\n   }\n  },\n  {\n   "short": "networkx",\n   "repo": "networkx/networkx",\n   "url": "https://github.com/networkx/networkx",\n   "pkg": "networkx",\n   "licence": "BSD-3-Clause",\n   "instance_ids": [\n    "networkx_8931",\n    "networkx_8906",\n    "networkx_8892",\n    "networkx_8890",\n    "networkx_8895",\n    "networkx_8899",\n    "networkx_8883",\n    "networkx_8882",\n    "networkx_8881",\n    "networkx_8880",\n    "networkx_8879",\n    "networkx_8620",\n    "networkx_8837",\n    "networkx_8813",\n    "networkx_8800",\n    "networkx_8808",\n    "networkx_8807",\n    "networkx_8750",\n    "networkx_8734",\n    "networkx_8726",\n    "networkx_8659"\n   ],\n   "fix_commits": {\n    "networkx_8931": "c1ebe046ec672eca7cba4dc88d6e0145406d693a",\n    "networkx_8906": "9f6079c0120416599ea5d4018bbaba229ea2e456",\n    "networkx_8892": "083e47255d7087340e97ec13d256e9241ec25e77",\n    "networkx_8890": "c94928ed94899033126c9d47f797a1f698584b20",\n    "networkx_8895": "56c7b0d0b6141ec0ffa6a9ebb5494417120b7f50",\n    "networkx_8899": "31ab6fd7e1e149e887282b83a91752c486d071b6",\n    "networkx_8883": "0db8227000872d7a9f6ce84c54ba1e5e99429122",\n    "networkx_8882": "8e0b38e1a11335f07db13a13a6a5955eecf33408",\n    "networkx_8881": "c279073cc33dab38777e701370210d0a7a98849a",\n    "networkx_8880": "0e3f58edd891bbef8f8013f147e8b679deba9970",\n    "networkx_8879": "73cb92bbaed12828848f2609340442119e7e0fe8",\n    "networkx_8620": "0c0c31bc54c563bb31771ef6420454a697aed12c",\n    "networkx_8837": "416c3e26da0567c9242aa415810dcffca27d625f",\n    "networkx_8813": "2aecdbdd18210c38dc2b9fe32da3825bf30a3c33",\n    "networkx_8800": "139259c6c5479b35ddde9318203ae715b25a3192",\n    "networkx_8808": "a5652a9d440b6d98ec7c88761834adc293680d42",\n    "networkx_8807": "2ebe5f80a114a181e03e18acc9491e45346598c0",\n    "networkx_8750": "b1f956344071ba1adfad17d56d62a7a78dfd37e6",\n    "networkx_8734": "d8081795e1344c576590b214c2e4f264b5e87244",\n    "networkx_8726": "e9eac8a7e04df0b9cdcff8ed1682b3971678f4ed",\n    "networkx_8659": "e187ba20b38583a329c960062e4b9e0f23c8fc74"\n   }\n  },\n  {\n   "short": "sympy",\n   "repo": "sympy/sympy",\n   "url": "https://github.com/sympy/sympy",\n   "pkg": "sympy",\n   "licence": "BSD-3-Clause",\n   "instance_ids": [\n    "sympy_30866",\n    "sympy_30141",\n    "sympy_30841",\n    "sympy_30646",\n    "sympy_30768",\n    "sympy_30716",\n    "sympy_29504",\n    "sympy_30591",\n    "sympy_30590",\n    "sympy_30567",\n    "sympy_30515",\n    "sympy_30476",\n    "sympy_30562",\n    "sympy_30514"\n   ],\n   "fix_commits": {\n    "sympy_30866": "a11605ca930cc3138cdd6c90bc8179e9e166dd8b",\n    "sympy_30141": "17594ab1954648db1238cb1372c50b479b70fc73",\n    "sympy_30841": "f8ea58b03ec08777ed7832bf85606c1fae9f935a",\n    "sympy_30646": "65c7920cb5e5c8958872c11f3ce1b3ba6c3e3395",\n    "sympy_30768": "12a149c2196ea036e3115fa723b20ed311499db3",\n    "sympy_30716": "bcd3a171919e08f940d5e97b165b5191f37ed901",\n    "sympy_29504": "2c89c30012477411849aa590861dfcae7d1e11f4",\n    "sympy_30591": "a1021ffb1f40f860960119e70e6329cd5937862c",\n    "sympy_30590": "94f14043f5195c6c58d2c21dcc2beccd9d476aa4",\n    "sympy_30567": "5c5ec9b873a29b2fdc8dc6759bb9a56f6b3d69c4",\n    "sympy_30515": "f14b5cd2d13c0145521eccec991c315794952117",\n    "sympy_30476": "ee699d48c852e1e98e719870c0dab6905e8326ed",\n    "sympy_30562": "5968b9bfd2bd3db4b6caf35bc07712cb70b8807b",\n    "sympy_30514": "31e03dc7ed8114117274ea679be1835044cec9c9"\n   }\n  },\n  {\n   "short": "pyparsing",\n   "repo": "pyparsing/pyparsing",\n   "url": "https://github.com/pyparsing/pyparsing",\n   "pkg": "pyparsing",\n   "licence": "MIT",\n   "instance_ids": [\n    "pyparsing_653",\n    "pyparsing_652",\n    "pyparsing_a6ae98ba59",\n    "pyparsing_92f52e2c3b",\n    "pyparsing_9b728b8835",\n    "pyparsing_e378aaf45c",\n    "pyparsing_16f348651c",\n    "pyparsing_73a8b553fb",\n    "pyparsing_621"\n   ],\n   "fix_commits": {\n    "pyparsing_653": "efd56db4e59f36b1673ce0eb0823e3afaa9d1201",\n    "pyparsing_652": "51560caba558d87349f4f497b82acef1de9ec8f8",\n    "pyparsing_a6ae98ba59": "a6ae98ba59b993d0b6d0475e7cb3b0a33c853f22",\n    "pyparsing_92f52e2c3b": "92f52e2c3b3ffb24215e966c0c349ca5678a7f86",\n    "pyparsing_9b728b8835": "9b728b8835aeef5974f0ee2be8df204b642ba008",\n    "pyparsing_e378aaf45c": "e378aaf45c5fad96d69daac25f53ff7469cb2878",\n    "pyparsing_16f348651c": "16f348651c167f7d557230ecbdf5a678c62e72aa",\n    "pyparsing_73a8b553fb": "73a8b553fb5fc50b3cb2e4b5b2b8652f8d997a29",\n    "pyparsing_621": "0a2d906a3e6b9a30b6a0701588caec3937ff271f"\n   }\n  },\n  {\n   "short": "tomlkit",\n   "repo": "python-poetry/tomlkit",\n   "url": "https://github.com/python-poetry/tomlkit",\n   "pkg": "tomlkit",\n   "licence": "MIT",\n   "instance_ids": [\n    "tomlkit_565",\n    "tomlkit_642",\n    "tomlkit_599",\n    "tomlkit_641",\n    "tomlkit_610",\n    "tomlkit_620",\n    "tomlkit_600",\n    "tomlkit_550",\n    "tomlkit_551",\n    "tomlkit_549",\n    "tomlkit_545",\n    "tomlkit_533",\n    "tomlkit_530",\n    "tomlkit_532",\n    "tomlkit_526",\n    "tomlkit_498",\n    "tomlkit_500",\n    "tomlkit_499",\n    "tomlkit_486",\n    "tomlkit_477"\n   ],\n   "fix_commits": {\n    "tomlkit_565": "229ee8660872df482a56c93c83b14ac98fedf864",\n    "tomlkit_642": "c07c0c15c49a893bb8a4c9f63401822d31e08942",\n    "tomlkit_599": "8c30fc17d91aee7469c0287e509300e78aa15227",\n    "tomlkit_641": "01b0f45cd43ca6d250e733b2db3a2ac84426c055",\n    "tomlkit_610": "f466807e253b7479a3dbd9621f8b0782384ade9e",\n    "tomlkit_620": "49d6efa980fedb321f16371a4b05abd2a449f775",\n    "tomlkit_600": "4b38becd75405e0f38ef3e47e9839eb407591c8a",\n    "tomlkit_550": "495a42ecc9119eaaeb895def0fd025a71cd1cf60",\n    "tomlkit_551": "8cd44f58b499da3c80a29bc877a512665a15e0bb",\n    "tomlkit_549": "67d3e86502706df2a9ed67114ab67e25bcdcbbb7",\n    "tomlkit_545": "e23a25438c6cc9b8cc28b6dcf76dc52f9c7a33e3",\n    "tomlkit_533": "43668ddebc3f082bf385d328aceed18d26976897",\n    "tomlkit_530": "d3c76f0bbe90af4b12a3b83bd8e59bc9061dfe85",\n    "tomlkit_532": "11e22aefccd8069a90ae75d20613b9b1068754a1",\n    "tomlkit_526": "ee4eedbacc8eb72872af050996c7396ecce190b9",\n    "tomlkit_498": "de68fd06eef61aab0d50d69fb589a43f00a046bf",\n    "tomlkit_500": "a076a6afbe035694bd6b1cfdbd8ad7284e93b862",\n    "tomlkit_499": "f8ce14b39c3bfe57f20f75fbc29cb9e5586811e7",\n    "tomlkit_486": "568739658e92b2836c4ead6dc029482f76bbfee8",\n    "tomlkit_477": "1d499870617c8e55b9322f99a2d64a7670f3b929"\n   }\n  }\n ],\n "d1_d2_exclusions": {\n  "sympy/sympy": "graph > 20,000 nodes: harness get_graph is O(N^2) above ~25k nodes (embedding LRU of 50,000); stays in the export and the claim-5 audit"\n }\n}'); print('wrote pilot_d/manifest.json')

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/pilot_d/env', exist_ok=True)
open('/tmp/d0a/pd/pilot_d/env/_harness.txt', 'w').write('# What the harness itself needs in the host env to run verification (Dockerfile.sandbox pins pytest-timeout 2.1.0)\npytest==9.1.1\npytest-timeout==2.1.0\n'); print('wrote pilot_d/env/_harness.txt')

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/pilot_d/env', exist_ok=True)
open('/tmp/d0a/pd/pilot_d/env/networkx.txt', 'w').write('# Test-only dependencies of networkx/networkx used via pytest.importorskip (requirements/test.txt + extras)\nnumpy==2.3.4\nscipy==1.16.3\npandas==2.3.3\n'); print('wrote pilot_d/env/networkx.txt')

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/pilot_d/env', exist_ok=True)
open('/tmp/d0a/pd/pilot_d/env/pyparsing.txt', 'w').write('# pyparsing/pyparsing: no test-only dependencies\n'); print('wrote pilot_d/env/pyparsing.txt')

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/pilot_d/env', exist_ok=True)
open('/tmp/d0a/pd/pilot_d/env/sqlglot.txt', 'w').write('# Test-only dependencies of tobymao/sqlglot (from setup.py extras "dev", imported by tests/)\nduckdb==1.4.1\npandas==2.3.3\npytz==2025.2\npython-dateutil==2.9.0.post0\nsix==1.17.0\ntzdata==2025.2\n'); print('wrote pilot_d/env/sqlglot.txt')

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/pilot_d/env', exist_ok=True)
open('/tmp/d0a/pd/pilot_d/env/sympy.txt', 'w').write('# Test-only dependencies of sympy/sympy (install_requires + sympy/conftest.py requires hypothesis)\nmpmath==1.3.0\nhypothesis==6.140.2\nattrs==25.4.0\nsortedcontainers==2.4.0\n'); print('wrote pilot_d/env/sympy.txt')

In [ ]:
import os; os.makedirs('/tmp/d0a/pd/pilot_d/env', exist_ok=True)
open('/tmp/d0a/pd/pilot_d/env/tomlkit.txt', 'w').write('# python-poetry/tomlkit: no test-only dependencies (tests/toml-test is a git submodule, absent from snapshots)\n'); print('wrote pilot_d/env/tomlkit.txt')

## 1. Task pool: clone, mine, build snapshots (public GitHub)

In [ ]:
os.chdir(PD)
man = json.load(open('pilot_d/manifest.json'))
if DRY:
    keep = os.environ.get('D0_DRYRUN_REPOS')
    man['repos'] = [r for r in man['repos'] if not keep or r['short'] in keep.split(',')]
    for r in man['repos']: r['instance_ids'] = r['instance_ids'][:2]
    print('DRY RUN: 2 tasks per repo', [r['short'] for r in man['repos']])
P = OUT / 'pool'; t0 = time.time()
for repo in man['repos']:
    d = TMP / 'src' / repo['short']
    if not d.exists():
        subprocess.run(['git', 'clone', '-q', f"--shallow-since={man['clone_shallow_since']}", repo['url'], str(d)], check=True)
    cand = P / f"cand_{repo['short']}.jsonl"
    subprocess.run([sys.executable, 'common/mining.py', str(d), '--repo', repo['repo'], '--pkg', repo['pkg'],
                    '--since', man['since'], '--out', str(cand), '--funnel', str(P / f"funnel_{repo['short']}.json")], check=True)
    subprocess.run([sys.executable, 'pilot_d/build_tasks.py', str(cand), str(d), str(P), '--ids', *repo['instance_ids']], check=True)
n_built = sum(1 for _ in open(P / 'tasks.jsonl'))
print('built', n_built, 'of', sum(len(r['instance_ids']) for r in man['repos']), 'minutes', round((time.time() - t0) / 60, 1))

## 2. Issue text (GitHub API; issue first, then an issue linked from the PR body, then the PR body)

In [ ]:
if DRY:  # the session that dry-runs this notebook cannot reach api.github.com
    shutil.copy(P / 'tasks.jsonl', P / 'tasks_text.jsonl'); shutil.copy(P / 'meta.jsonl', P / 'meta_text.jsonl')
else:
    subprocess.run([sys.executable, 'common/github_text.py', str(P / 'tasks.jsonl'), str(P / 'meta.jsonl'),
                    str(P / 'tasks_text.jsonl'), str(P / 'meta_text.jsonl')], check=True)
src = [json.loads(l).get('problem_statement_source', 'pending') for l in open(P / 'meta_text.jsonl')]
print({k: sum(s.split('#')[0] == k for s in src) for k in ('issue', 'pr_body', 'pending')})

## 3. Wheels for D0b's offline install (exactly what this image lacks)

In [ ]:
MODULE_OF = {'python-dateutil': 'dateutil', 'pytest-timeout': 'pytest_timeout'}
missing = []
for env in sorted(glob.glob('pilot_d/env/*.txt')):
    for line in open(env):
        line = line.split('#')[0].strip()
        if line and importlib.util.find_spec(MODULE_OF.get(line.split('==')[0], line.split('==')[0])) is None:
            missing.append(line)
need = ['sentence-transformers==6.2.0'] + missing
(OUT / 'need.txt').write_text('\n'.join(need) + '\n')
subprocess.run([*PIP, 'install', '--dry-run', '--quiet', '--report', str(TMP / 'report.json'), '-r', str(OUT / 'need.txt')], check=True)
pins = sorted({f"{i['metadata']['name']}=={i['metadata']['version']}" for i in json.load(open(TMP / 'report.json'))['install']})
for pin in pins:  # wheels only; build one from sdist if a project has no wheel
    if subprocess.run([*PIP, 'download', '-q', '--no-deps', '--only-binary=:all:', '-d', str(OUT / 'wheels'), pin]).returncode != 0:
        subprocess.run([*PIP, 'wheel', '-q', '--no-deps', '-w', str(OUT / 'wheels'), pin], check=True)
subprocess.run([*PIP, 'wheel', '-q', '--no-deps', '-w', str(OUT / 'wheels'), 'git+https://github.com/danielmantiglia/codegraph-loc@1e8e0abbb97a1f78648cb12890fd700712b00623'], check=True)
for env in sorted(glob.glob('pilot_d/env/*.txt')):  # test-only wheels shipped inside the export (both sandbox Pythons)
    if not any(l.split('#')[0].strip() for l in open(env)): continue
    for pyv in ('3.12', '3.13'):
        subprocess.run([*PIP, 'download', '-q', '--no-deps', '-r', env, '--only-binary=:all:', '--python-version', pyv,
                        '--platform', 'manylinux2014_x86_64', '--platform', 'manylinux_2_17_x86_64',
                        '--platform', 'manylinux_2_28_x86_64', '-d', str(OUT / 'export_wheels')], check=True)
print('need:', need); print(len(pins), 'pinned distributions;', len(list((OUT / 'wheels').glob('*.whl'))), 'wheels')

## 4. Model files, then a self-check: offline install from `wheels/` + offline embedding

In [ ]:
subprocess.run([*PIP, 'install', '-q', '--no-index', '--find-links', str(OUT / 'wheels'), '-r', str(OUT / 'need.txt')], check=True)
from huggingface_hub import snapshot_download
MD = OUT / 'models' / 'bge-small-en-v1.5'
snapshot_download('BAAI/bge-small-en-v1.5', local_dir=str(MD), allow_patterns=['*.json', '*.txt', 'model.safetensors', '1_Pooling/*'])
check = subprocess.run([sys.executable, '-c', (
    "import os; os.environ['HF_HUB_OFFLINE']='1'; os.environ['TRANSFORMERS_OFFLINE']='1'\n"
    "from sentence_transformers import SentenceTransformer\n"
    f"m = SentenceTransformer('{MD}', trust_remote_code=False, device='cpu', model_kwargs={{'use_safetensors': True}})\n"
    "v = m.encode(['def f(x): return x'], normalize_embeddings=True); print('offline embedding ok', v.shape)")],
    capture_output=True, text=True)
print(check.stdout.strip(), check.stderr[-500:]); assert 'offline embedding ok' in check.stdout

## 5. Manifest (sha256 per file) and summary

In [ ]:
files = {}
for f in sorted(OUT.rglob('*')):
    if f.is_file() and f.name != 'D0A_MANIFEST.json':
        h = hashlib.sha256()
        with open(f, 'rb') as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b''): h.update(chunk)
        files[str(f.relative_to(OUT))] = {'sha256': h.hexdigest(), 'bytes': f.stat().st_size}
meta = [json.loads(l) for l in open(P / 'meta_text.jsonl')]
summary = {'version': 1, 'built': n_built, 'need': need, 'pinned_wheels': pins, 'dry_run': DRY,
           'by_repo': {r['repo']: sum(m['repo'] == r['repo'] for m in meta) for r in man['repos']},
           'problem_statement_source': {k: sum((m.get('problem_statement_source') or 'pending').split('#')[0] == k for m in meta)
                                        for k in ('issue', 'pr_body', 'pending')},
           'n_files': len(files) + 2, 'gigabytes': round(sum(v['bytes'] for v in files.values()) / 1e9, 2),
           'minutes': round((time.time() - t_start) / 60, 1)}
json.dump({'files': files, 'summary': summary}, open(OUT / 'D0A_MANIFEST.json', 'w'))
json.dump(summary, open(OUT / 'D0a_summary.json', 'w'), indent=1); print(json.dumps(summary, indent=1)[:1500])